In [26]:
import numpy as np
import networkx as nx

def find_k3(G):
    triangles_list = set()
    triangles = [clique for clique in nx.enumerate_all_cliques(G) if len(clique) == 3]
    for triangle in triangles:
        triangles_list.add(tuple(sorted(triangle)))
    triangles_list = [list(tri) for tri in triangles_list]

    return triangles_list

def deleteNotUsedRois(adj_matrix):
    roi_no_to_delete = [81, 80, 35, 34]

    for roi_no in roi_no_to_delete:
        adj_matrix = np.delete(adj_matrix, roi_no, axis=0)
        adj_matrix = np.delete(adj_matrix, roi_no, axis=1)

    return adj_matrix

def import_connectome(connectome_path, no_weights):
    adj_matrix = np.loadtxt(connectome_path, delimiter=',', dtype=float)

    adj_matrix = deleteNotUsedRois(adj_matrix)

    if no_weights:
        adj_matrix[adj_matrix != 0.0] = 1.0

    G = nx.from_numpy_array(adj_matrix)
    triangles_list = find_k3(G)

    return adj_matrix, triangles_list

In [27]:
baseline_data_dir = '/Volumes/External/ADNI_derivatives/sub-AD4009/ses-baseline/'
followup_data_dir = '/Volumes/External/ADNI_derivatives/sub-AD4009/ses-followup/'
connect_matrix_filename = 'dwi/connect_matrix_norm.csv'
baseline_AB_lvl_filename = 'pet/sub-AD4009_ses-baseline_acq-AP_date-2011-07-07_trc-av45_pet.csv'
followup_AB_lvl_filename = "pet/sub-AD4009_ses-followup_acq-AP_date-2013-07-03_trc-av45_pet.csv"

adj_matrix_raw, triangles_list = import_connectome(baseline_data_dir + connect_matrix_filename, True)
baseline_AB_lvl = np.loadtxt(baseline_data_dir+baseline_AB_lvl_filename, delimiter=',', dtype=float)
followup_AB_lvl = np.loadtxt(followup_data_dir+followup_AB_lvl_filename, delimiter=',', dtype=float)

In [31]:
#!pip install torch-geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import HypergraphConv
from torch_geometric.nn import MessagePassing

adj_matrix = torch.from_numpy(adj_matrix_raw)
hyperedges = triangles_list

# Step 3: Convert cliques into edge_index format
num_nodes = adj_matrix.size(0)
num_hyperedges = len(hyperedges)
row = torch.tensor([node for clique in hyperedges for node in clique])  # Nodes
col = torch.arange(num_hyperedges).repeat_interleave(torch.tensor([len(clique) for clique in hyperedges])) + num_nodes
 # Virtual hyperedge nodes
edge_index = torch.stack([row, col], dim=0)  # Shape [2, num_edges]

# Step 4: Define features and prepare the data
x_input = torch.cat([torch.from_numpy(baseline_AB_lvl.reshape(-1, 1)), torch.zeros((num_hyperedges, 1))], dim=0).float()  # Include virtual nodes
y = torch.from_numpy(followup_AB_lvl.reshape(-1, 1)).float()
data = Data(x=x_input, edge_index=edge_index, y=y)

# Step 5: Define and train the model
class HypergraphNet(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(HypergraphNet, self).__init__()
        self.conv1 = HypergraphConv(in_channels, hidden_channels)
        self.conv2 = HypergraphConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x
    
# Model setup
in_channels = 1
hidden_channels = 16
out_channels = 1
model = HypergraphNet(in_channels, hidden_channels, out_channels)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.mse_loss(out[:num_nodes], data.y)  # Exclude virtual nodes
    loss.backward()
    optimizer.step()
    return loss.item()

def test():
    model.eval()
    out = model(data.x, data.edge_index)
    mse = F.mse_loss(out[:num_nodes], data.y).item()
    return mse

for epoch in range(1, 101):
    loss = train()
    mse = test()
    if epoch % 10 == 0:
        print(f"Epoch: {epoch:03d}, Loss: {loss:.4f}, Test MSE: {mse:.4f}")

# Final predictions
model.eval()
predicted_features = model(data.x, data.edge_index)[:num_nodes]
print("Predicted features at nodes:")
print(predicted_features)


Epoch: 010, Loss: 0.0307, Test MSE: 0.0282
Epoch: 020, Loss: 0.0318, Test MSE: 0.0311
Epoch: 030, Loss: 0.0223, Test MSE: 0.0219
Epoch: 040, Loss: 0.0205, Test MSE: 0.0202
Epoch: 050, Loss: 0.0176, Test MSE: 0.0174
Epoch: 060, Loss: 0.0159, Test MSE: 0.0158
Epoch: 070, Loss: 0.0144, Test MSE: 0.0142
Epoch: 080, Loss: 0.0131, Test MSE: 0.0130
Epoch: 090, Loss: 0.0121, Test MSE: 0.0120
Epoch: 100, Loss: 0.0113, Test MSE: 0.0112
Predicted features at nodes:
tensor([[0.3479],
        [0.3454],
        [0.3482],
        [0.3469],
        [0.3478],
        [0.3459],
        [0.3492],
        [0.3449],
        [0.3483],
        [0.3442],
        [0.3483],
        [0.3462],
        [0.3497],
        [0.3458],
        [0.3450],
        [0.3454],
        [0.3469],
        [0.3458],
        [0.3476],
        [0.3480],
        [0.3478],
        [0.3485],
        [0.3476],
        [0.3469],
        [0.3467],
        [0.3459],
        [0.3475],
        [0.3455],
        [0.3469],
        [0.3445],
 